In [1]:
# function to solve B using non-negative least squares
from scipy.optimize import nnls
%cd ..
from compiler_options import compiler_decorator
from myutils import khatri_rao_power, khatri_rao_product, \
     option_parser, compiler_decorator, prange, pos, isbool, \
    dormqr, lapack, norm

from part_sym_SPM import spm_21sym

from matplotlib import pyplot as plt
import seaborn as sns
import numpy as np
from scipy.linalg import svd

import numpy as np
# the output of this cell is covid_T.npy
import anndata as ad
from sklearn.decomposition import PCA
import pandas as pd


def nnls_refine_B(T, A):
    """
    Refine B matrix using non-negative least squares.
    
    Parameters:
    -----------
    T : numpy.ndarray
        Original tensor (m × m × n)
    A : numpy.ndarray  
        A matrix from decomposition (m × r)
        
    Returns:
    --------
    B_nnls : numpy.ndarray
        Non-negative refined B matrix
    """
    m, _, n = T.shape
    r = A.shape[1]
    
    # Reshape tensor for matrix operations
    T_mat = T.reshape(m, -1)  # m × (m*n)
    
    # Compute Khatri-Rao product A ⊙ A
    A_kr = khatri_rao_power(A, 2)  # (m²) × r
    
    B_nnls = np.zeros((n, r))
    residual_list = []
    # Solve for each context (row of B) separately
    for i in range(n):
        # Extract the i-th slice: T[:,:,i]
        T_slice = T[:, :, i].flatten()  # Flatten to vector of length m²
        
        # Solve non-negative least squares: min ||A_kr * b_i - T_slice||²
        # subject to b_i >= 0
        b_i, residual = nnls(A_kr, T_slice)
        residual_list.append(residual)
        B_nnls[i, :] = b_i

    return B_nnls,residual_list

# A, B = spm_21sym(T_500,r)
# B_nnls, residual_list = nnls_refine_B(T_500.copy(), A)

/ewsc/exxact04/sbhate/mcpc/covid_code


In [2]:
import anndata as ad
from sklearn.decomposition import PCA
import pandas as pd
dataset = ad.read_h5ad('/ewsc/exxact04/sbhate/mcpc/spm-private/downstream/gwps_k562_all_raw.h5ad')


In [3]:
x_sub = dataset[dataset.obs[dataset.obs['UMI_count']>2000].index].to_df()
x_sub = np.log(1 + 1000* x_sub/x_sub.values.sum(axis = 1,keepdims=True))


In [4]:
pca = PCA(n_components=400, random_state=418)
pca.fit(x_sub.sample(500000, random_state = 4))
X_pc = pd.DataFrame(pca.transform(x_sub),index = x_sub.index)
# X_pc = x_sub
covs = X_pc.groupby(dataset.obs['gene'].astype(str).loc[X_pc.index]).apply(lambda w : np.cov(w.T))
means = X_pc.groupby(dataset.obs['gene'].astype(str).loc[X_pc.index]).apply(lambda w : np.mean(w,axis=0))
stds = X_pc.groupby(dataset.obs['gene'].astype(str).loc[X_pc.index]).apply(lambda w : np.std(w,axis=0))
T_500 = np.stack(covs,axis = 2)

In [5]:
counts = dataset.obs['gene'].astype(str).loc[X_pc.index].value_counts()


In [37]:
!pwd

/ewsc/exxact04/sbhate/mcpc


In [6]:
context_counts = pd.read_pickle('covid_code/covid/context_counts.pkl')
B = np.load('covid_code/covid/perturbseq_B.npy')
B_nnls = np.load('covid_code/covid/perturbseq_B_nnls.npy')
A = np.load('covid_code/covid/perturbseq_A.npy')
B_pc = means.loc[context_counts.index].iloc[:,:128]
B_mc = pd.DataFrame(B,index = context_counts.index)
B_mc_nnls= pd.DataFrame(B_nnls,index = context_counts.index)
B_pc_std = stds.loc[context_counts.index].iloc[:,:128]

FileNotFoundError: [Errno 2] No such file or directory: 'covid_code/covid/context_counts.pkl'

In [362]:
gi = pd.Series(B_nnls.mean(axis = 0)).sort_values().iloc[-40:].index

In [6]:
corum = pd.read_table('covid/corum_humanComplexes.txt')

In [7]:
signor = pd.read_table('covid/all_data_10_11_25.tsv',sep='\t')

In [9]:
sub_covs = {k:v[:100,:100] for k,v in covs.items() if counts.loc[k]>300}
T_500_2 = np.stack(pd.Series(sub_covs),axis = 2)

Bs = {}
for r in [5,10,20,30,40,50]:
    if r in Bs:
        continue
    A, B = spm_21sym(T_500_2,r)
    B_nnls, residual_list= nnls_refine_B(T_500_2.copy(), A)

    B_mc= pd.DataFrame(B, index = sub_covs.keys())
    B_mc_nnls = pd.DataFrame(B_nnls, index = sub_covs.keys())
    Bs[r] = (B_mc.copy(), B_mc_nnls.copy())

In [10]:
links_signor = set()
for i,j in zip(signor['ENTITYA'],signor['ENTITYB']):
    if (i in B_mc.index) and (j in B_mc.index):
        links_signor.add(tuple(sorted([i,j])))


In [11]:
import networkx as nx
import itertools
g = nx.Graph()

links_corum = set()
for geneset in corum['subunits_gene_name']:
        for i,j in itertools.combinations(geneset.split(';'),2):
            if (i in B_mc.index) and (j in B_mc.index):
                links_corum.add(tuple(sorted([i,j])))

In [20]:

links_go5 = set()
for _, geneset in {k:B_mc.index[v] for k,v in go_inds.items() if (len(v)>0) & (len(v)<=5)}.items():
        for i,j in itertools.combinations(geneset,2):
            if (i in B_mc.index) and (j in B_mc.index):
                links_go5.add(tuple(sorted([i,j])))

links_go10 = set()
for _, geneset in {k:B_mc.index[v] for k,v in go_inds.items() if (len(v)>0) & (len(v)<=10)}.items():
        for i,j in itertools.combinations(geneset,2):
            if (i in B_mc.index) and (j in B_mc.index):
                links_go10.add(tuple(sorted([i,j])))

links_go20 = set()
for _, geneset in {k:B_mc.index[v] for k,v in go_inds.items() if (len(v)>0) & (len(v)<=20)}.items():
        for i,j in itertools.combinations(geneset,2):
            if (i in B_mc.index) and (j in B_mc.index):
                links_go20.add(tuple(sorted([i,j])))

links_gwas5 = set()
for _, geneset in {k:B_mc.index[v] for k,v in gwas_inds.items() if (len(v)>0) & (len(v)<=5)}.items():
        for i,j in itertools.combinations(geneset,2):
            if (i in B_mc.index) and (j in B_mc.index):
                links_gwas5.add(tuple(sorted([i,j])))

links_gwas10 = set()
for _, geneset in {k:B_mc.index[v] for k,v in gwas_inds.items() if (len(v)>0) & (len(v)<=10)}.items():
        for i,j in itertools.combinations(geneset,2):
            if (i in B_mc.index) and (j in B_mc.index):
                links_gwas10.add(tuple(sorted([i,j])))

links_gwas20 = set()
for _, geneset in {k:B_mc.index[v] for k,v in gwas_inds.items() if (len(v)>0) & (len(v)<=20)}.items():
        for i,j in itertools.combinations(geneset,2):
            if (i in B_mc.index) and (j in B_mc.index):
                links_gwas20.add(tuple(sorted([i,j])))

links_cell5 = set()
for _, geneset in {k:B_mc.index[v] for k,v in cell_inds.items() if (len(v)>0) & (len(v)<=5)}.items():
        for i,j in itertools.combinations(geneset,2):
            if (i in B_mc.index) and (j in B_mc.index):
                links_cell5.add(tuple(sorted([i,j])))

links_cell10 = set()
for _, geneset in {k:B_mc.index[v] for k,v in cell_inds.items() if (len(v)>0) & (len(v)<=10)}.items():
        for i,j in itertools.combinations(geneset,2):
            if (i in B_mc.index) and (j in B_mc.index):
                links_cell10.add(tuple(sorted([i,j])))

links_cell20 = set()
for _, geneset in {k:B_mc.index[v] for k,v in cell_inds.items() if (len(v)>0) & (len(v)<=20)}.items():
        for i,j in itertools.combinations(geneset,2):
            if (i in B_mc.index) and (j in B_mc.index):
                links_cell20.add(tuple(sorted([i,j])))

In [21]:
from scipy.spatial.distance import pdist,cdist 

In [22]:
def znorm(df,axis=0):
    return (df - df.values.mean(axis = axis,keepdims=True))/(1e-4 + df.values.std(axis = axis,keepdims=True))

In [23]:
def get_recall(labelled_B,links,normalize= True,metric = 'cosine'):
    if normalize:
        z = znorm(labelled_B)
    else:
        z = labelled_B
    top_dist = cdist(z,z,metric= metric)
    trinds = np.triu_indices(len(top_dist))
    pdist = top_dist[trinds]
    min_c, max_c = np.percentile(top_dist[trinds], 5), np.percentile(top_dist[trinds], 95)
    ii = z.index.values[trinds[0][np.where((pdist <min_c)| (pdist >max_c))[0]]]
    jj = z.index.values[trinds[1][np.where((pdist <min_c)| (pdist >max_c))[0]]]
    preds = set()
    for g1,g2 in zip(ii,jj):
        preds.add(tuple(sorted([g1,g2])))

    return len(links.intersection(preds))/len(links)

In [302]:
get_recall(pd.concat([0.00005*znorm(Bs[40][1]), ],axis=1),links_signor)

0.15985130111524162

In [303]:

get_recall(means.loc[sub_covs.keys()].iloc[:,:40],links_react,metric = 'cosine',normalize =True)

0.1819336402603588

In [26]:
recalls = {}
for name,links in zip(['signor','corum','react','go5','go10', 'go20', 'gwas5','gwas10','gwas20','cell5','cell10','cell20'],[links_signor,links_corum,links_react,links_go5,links_go10,links_go20,links_gwas5,links_gwas10,links_gwas20, links_cell5, links_cell10,links_cell20]):
    for r in sorted(Bs.keys()):
        print(r)
        # print(get_recall(Bs[r][0],links,metric = 'cosine',normalize =True))
        # print(get_recall(stds.loc[sub_covs.keys()].iloc[:,:r],links,metric = 'cosine',normalize =True))
        recalls[name,r,'mean'] = get_recall(means.loc[sub_covs.keys()],links,metric = 'cosine',normalize =True)
        recalls[name,r,'cat_mc'] = get_recall(pd.concat([znorm(Bs[r][0]),znorm(means.loc[sub_covs.keys()]) ],axis=1),links,metric = 'cosine',normalize =True)
        recalls[name,r,'cat_pcstd'] = get_recall(pd.concat([znorm(stds.loc[sub_covs.keys()].iloc[:,:r]**2),znorm(means.loc[sub_covs.keys()]) ],axis=1),links,metric = 'cosine',normalize =True)
        
        

5
10
20
30
40
50
5
10
20
30
40
50
5
10
20
30
40
50
5
10
20
30
40
50
5
10
20
30
40
50
5
10
20
30
40
50
5
10
20
30
40
50
5
10
20
30
40
50
5
10
20
30
40
50
5
10
20
30
40
50
5
10
20
30
40
50
5
10
20
30
40
50


In [27]:
recall_table = pd.Series(recalls).unstack().reset_index().groupby('level_0').apply(lambda w:
                                                                    pd.Series({'cat_mc': w['cat_mc'].max(), 'best_rank_mc': w['level_1'].iloc[w['cat_mc'].argmax()],
                                                                    'cat_pc': w['cat_pcstd'].max(), 'best_rank_pcstd': w['level_1'].iloc[w['cat_pcstd'].argmax()],
                                                                               'mean': w['mean'].iloc[0]})).round(4)
                                                                   

/tmp/ipykernel_2718374/1889704631.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  recall_table = pd.Series(recalls).unstack().reset_index().groupby('level_0').apply(lambda w:


In [28]:
supp_table = recall_table.loc[['cell5','cell10', 'cell20', 'go5','go10', 'go20', 'gwas5', 'gwas10',
       'gwas20',  'react', 'signor','corum']].round(4)

In [29]:
supp_table['best_rank_mc'] =supp_table['best_rank_mc'] .astype(int) 
supp_table['best_rank_pcstd'] =supp_table['best_rank_pcstd'] .astype(int) 

In [30]:
supp_table

,cat_mc,best_rank_mc,cat_pc,best_rank_pcstd,mean
level_0,,,,,
cell5,0.0903,50,0.0799,10,0.0816
cell10,0.0951,5,0.0945,5,0.0963
cell20,0.0967,5,0.0978,20,0.0976
go5,0.1171,30,0.1174,50,0.1126
go10,0.1214,40,0.1199,50,0.1170
go20,0.1194,40,0.1168,20,0.1160
gwas5,0.0951,5,0.0944,5,0.0907
gwas10,0.0977,50,0.0951,50,0.0931
gwas20,0.0981,10,0.0972,5,0.0970


In [476]:
print(supp_table.to_latex(float_format = '%.4f'))

\begin{tabular}{lrrrrr}
\toprule
 & cat_mc & best_rank_mc & cat_pc & best_rank_pcstd & mean \\
level_0 &  &  &  &  &  \\
\midrule
cell5 & 0.0920 & 50 & 0.0781 & 5 & 0.0816 \\
cell10 & 0.0951 & 5 & 0.0951 & 5 & 0.0963 \\
cell20 & 0.0967 & 5 & 0.0982 & 5 & 0.0978 \\
go5 & 0.1163 & 40 & 0.1171 & 50 & 0.1129 \\
go10 & 0.1209 & 40 & 0.1200 & 50 & 0.1169 \\
go20 & 0.1180 & 40 & 0.1172 & 20 & 0.1160 \\
gwas5 & 0.0951 & 5 & 0.0951 & 5 & 0.0907 \\
gwas10 & 0.0969 & 50 & 0.0956 & 5 & 0.0933 \\
gwas20 & 0.0985 & 10 & 0.0972 & 5 & 0.0977 \\
react & 0.1915 & 30 & 0.1907 & 50 & 0.1797 \\
signor & 0.2268 & 5 & 0.2156 & 10 & 0.2082 \\
corum & 0.3075 & 20 & 0.3060 & 20 & 0.2902 \\
\bottomrule
\end{tabular}



In [437]:
pd.Series(recalls).unstack().reset_index().groupby('level_0').apply(lambda w:
                                                                    pd.Series({'cat_mc': w['cat_mc'].max(), 'best_rank_mc': w['level_1'].iloc[w['cat_mc'].argmax()],
                                                                    'cat_pc': w['cat_pcstd'].max(), 'best_rank_pcstd': w['level_1'].iloc[w['cat_pcstd'].argmax()],
                                                                               'mean': w['mean'].iloc[0]})).round(4)
                                                                   

/tmp/ipykernel_361754/3353956071.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  pd.Series(recalls).unstack().reset_index().groupby('level_0').apply(lambda w:


,cat_mc,best_rank_mc,cat_pc,best_rank_pcstd,mean
level_0,,,,,
cell,0.0967,5.0,0.0982,5.0,0.0978
corum,0.3075,20.0,0.3060,20.0,0.2902
go,0.1180,40.0,0.1172,20.0,0.1160
gwas,0.0985,10.0,0.0972,5.0,0.0977
react,0.1915,30.0,0.1907,50.0,0.1797
signor,0.2268,5.0,0.2156,10.0,0.2082


In [25]:
ne = pd.read_table('covid/NCBI2Reactome_PE_Pathway.txt',header = None)
ne['gene'] = [a[0] for a in ne[2].str.split(' ')]
sub_ne = ne[ne['gene'].isin(B_mc.index)]
links_react = set()
for k,v in sub_ne.groupby(3).groups.items():
    for i,j in itertools.combinations(ne[2].loc[v], 2):
        ia = i.split(' ')[0]
        ja = j.split(' ')[0]        
        links_react.add(tuple(sorted([ia,ja])))


/tmp/ipykernel_2718374/966820136.py:1: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  ne = pd.read_table('covid/NCBI2Reactome_PE_Pathway.txt',header = None)


In [13]:
with open('covid/GWAS_Catalog_2025') as f:
    lines = f.readlines()

gwas = {}
for line in lines:
    a = line.split('\t')
    gwas[a[0]] = set([b for b in a[1:] if b in B_mc.index ])

gwas = {k: list([a for a in v if a in B_mc.index]) for k,v in gwas.items()}
gwas_inds = {k : np.where(np.isin(B_mc.index,v))[0] for k,v in gwas.items()}

In [14]:
with open('covid/Human_Phenotype_Ontology') as f:
    lines = f.readlines()

hpi = {}
for line in lines:
    a = line.split('\t')
    hpi[a[0]] = set([b for b in a[1:] if b in B_mc.index ])

hpi = {k: list([a for a in v if a in B_mc.index]) for k,v in hpi.items()}
hpi_inds = {k : np.where(np.isin(B_mc.index,v))[0] for k,v in hpi.items()}

In [15]:
with open('covid/GO_Biological_Process_2025') as f:
    lines = f.readlines()

go = {}
for line in lines:
    a = line.split('\t')
    go[a[0]] = set([b for b in a[1:] if b in B_mc.index ])

go = {k: list([a for a in v if a in B_mc.index]) for k,v in go.items()}
go_inds = {k : np.where(np.isin(B_mc_nnls.index,v))[0] for k,v in go.items()}

In [16]:
with open('covid/MSigDB_Hallmark_2020') as f:
    lines = f.readlines()

go = {}
for line in lines:
    a = line.split('\t')
    go[a[0]] = set([b for b in a[1:] if b in B_mc.index ])

go = {k: list([a for a in v if a in B_mc.index]) for k,v in go.items()}
msigdb_inds = {k : np.where(np.isin(B_mc.index,v))[0] for k,v in go.items()}

In [17]:
with open('covid/ChEA_2022') as f:
    lines = f.readlines()

go = {}
for line in lines:
    a = line.split('\t')
    go[a[0]] = set([b for b in a[1:] if b in B_mc.index ])

go = {k: list([a for a in v if a in B_mc.index]) for k,v in go.items()}
chea_inds = {k : np.where(np.isin(B_mc.index,v))[0] for k,v in go.items()}

In [18]:
with open('covid/MGI_Mammalian_Phenotype_Level_4_2024') as f:
    lines = f.readlines()

go = {}
for line in lines:
    a = line.split('\t')
    go[a[0]] = set([b for b in a[1:] if b in B_mc.index ])

go = {k: list([a for a in v if a in B_mc.index]) for k,v in go.items()}
mgi_comp_inds = {k : np.where(np.isin(B_mc.index,v))[0] for k,v in go.items()}

In [19]:
with open('covid/CellMarker_2024') as f:
    lines = f.readlines()

go = {}
for line in lines:
    a = line.split('\t')
    go[a[0]] = set([b for b in a[1:] if b in B_mc.index ])

go = {k: list([a for a in v if a in B_mc.index]) for k,v in go.items()}
cell_inds = {k : np.where(np.isin(B_mc.index,v))[0] for k,v in go.items()}